## Init

In [37]:
import requests
import pandas as pd
from dotenv import dotenv_values
from pathlib import Path


In [38]:
# set pandas full outputs for testing
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

In [ ]:
ENV_FILE = dotenv_values(Path('secrets/.env'))

# TOKEN = ENV_FILE.get('TOKEN')
TOKEN = ENV_FILE.get('TOKEN_TEST')
HEADERS = {
    'Authorization': f'Bearer {TOKEN}',
    'Accept-Encoding': 'gzip',
}
LIMIT=100

# В Оркестраторе реализовать:
# В Prefect есть встроенное хранилище переменных. Создать переменную LAST_SUCCESSFUL_SYNC.
# Скрипт читает переменную.
# Если она не задана (или равна None), Prefect понимает, что это «холодный старт», и забирает всё.
# В конце успешного запуска скрипт обновляет эту переменную текущим временем.
START_TIMESTAMP = '2026-04-05 00:00:00'
STOP_TIMESTAMP  = '2026-04-07 00:00:00'

 ---

 ## Вспомогательные функции

 ---

In [40]:
def filter_and_rename_columns(df: pd.DataFrame, mapping: dict) -> pd.DataFrame:
    """
    1. Оставляет в df только те колонки, которые есть в ключах mapping.
    2. Переименовывает их в значения из mapping.
    """
    # Находим пересечение: только те ключи словаря, которые реально есть в DF
    existing_cols = [col for col in mapping.keys() if col in df.columns]
    
    # Сначала фильтруем (создаем копию с нужными колонками), 
    # а затем переименовываем их по словарю
    return df[existing_cols].rename(columns=mapping)

###### Отдельный блок для json_normalize_extended #######

def _expand_kv_cell(
    cell,
    name_key: str,
    value_key: str,
    sep: str,
    source_col: str,
    on_duplicate_key: str,
) -> dict:
    if not isinstance(cell, list):
        return {}
    result = {}
    seen: dict[str, int] = {}
    for item in cell:
        if not isinstance(item, dict):
            continue
        name = item.get(name_key)
        value = item.get(value_key)
        if name is None:
            continue
        if name in seen:
            if on_duplicate_key == "raise":
                raise ValueError(
                    f"Дублирующийся ключ {name!r} в kv-списке колонки "
                    f"{source_col!r}. Передайте on_duplicate_key='prefix' "
                    f"чтобы разрешить коллизию автоматически."
                )
            seen[name] += 1
            name = f"{name}_{seen[name]}"
        else:
            seen[name] = 1

        if isinstance(value, dict):
            for sub_key, sub_val in value.items():
                result[f"{name}{sep}{sub_key}"] = sub_val
        else:
            result[name] = value
    return result


def _first_list_value(series):
    for v in series:
        if isinstance(v, list):
            return v
    return None


def _is_kv_list(val, name_key: str, value_key: str) -> bool:
    return (
        isinstance(val, list)
        and bool(val)
        and isinstance(val[0], dict)
        and name_key in val[0]
        and value_key in val[0]
    )


def json_normalize_extended(
    data,
    record_path=None,
    meta=None,
    meta_prefix=None,
    record_prefix=None,
    errors="raise",
    sep="_",
    max_level=None,
    kv_name_key="name",
    kv_value_key="value",
    on_collision="raise",
) -> pd.DataFrame:
    """
    Расширение pd.json_normalize с поддержкой разворачивания kv-списков.

    Если значение ячейки после нормализации — список словарей вида
        [{"<kv_name_key>": "x", "<kv_value_key>": <val>}, ...]
    то каждый элемент становится отдельной колонкой:
      - скалярный val  → колонка "x" со значением val
      - dict val       → колонки "x<sep>sub_key" для каждого ключа словаря

    Количество строк не изменяется. Исходная kv-колонка удаляется.

    Параметры идентичны pd.json_normalize, плюс:

    kv_name_key : str, default "name"
        Ключ в словаре kv-списка, значение которого становится именем колонки.

    kv_value_key : str, default "value"
        Ключ в словаре kv-списка, значение которого становится значением ячейки.

    on_collision : {"raise", "prefix"}, default "raise"
        Поведение при конфликте имён — когда kv-имя совпадает с уже существующей
        колонкой DataFrame, или когда одно kv-имя встречается в списке дважды.

        "raise"  — бросить ValueError с указанием конфликтующих имён и колонки-
                   источника. Используйте этот режим чтобы отловить неожиданные
                   коллизии в данных.

        "prefix" — разрешить коллизию автоматически, добавив к конфликтующему
                   имени префикс из названия исходной kv-колонки через sep:
                     · "партия" конфликтует, источник "assortment_characteristics"
                       → новая колонка: "assortment_characteristics_партия"
                   Дубли внутри одной строки нумеруются начиная со второго
                   вхождения:
                     · первый  "партия" → "партия"  (или с prefix если коллизия)
                     · второй  "партия" → "партия_2" (или с prefix: "..._партия_2")

    Исключения:
        ValueError — если on_collision="raise" и обнаружена коллизия имён.
        ValueError — если on_collision передан с недопустимым значением.
    """
    if on_collision not in ("raise", "prefix"):
        raise ValueError(
            f"on_collision должен быть 'raise' или 'prefix', получено {on_collision!r}."
        )

    df = pd.json_normalize(
        data,
        record_path=record_path,
        meta=meta,
        meta_prefix=meta_prefix,
        record_prefix=record_prefix,
        errors=errors,
        sep=sep,
        max_level=max_level,
    )

    kv_cols = []
    extra_frames = []

    for col in df.columns:
        first_list = _first_list_value(df[col])
        if not _is_kv_list(first_list, kv_name_key, kv_value_key):
            continue
        kv_cols.append(col)
        flat_rows = [
            _expand_kv_cell(
                cell,
                name_key=kv_name_key,
                value_key=kv_value_key,
                sep=sep,
                source_col=col,
                on_duplicate_key=on_collision,
            )
            for cell in df[col]
        ]
        extra_frames.append(pd.DataFrame(flat_rows, index=df.index))

    if not kv_cols:
        return df

    remaining_cols = set(df.columns) - set(kv_cols)
    new_cols = {c for frame in extra_frames for c in frame.columns}
    collisions = remaining_cols & new_cols

    if collisions:
        if on_collision == "raise":
            raise ValueError(
                f"Имена из kv-списков конфликтуют с существующими колонками: "
                f"{sorted(collisions)}. Передайте on_collision='prefix' "
                f"чтобы разрешить коллизию автоматически."
            )
        # on_collision == "prefix": переименовываем конфликтующие колонки
        # в каждом extra_frame, добавляя префикс из имени исходной kv-колонки
        renamed_frames = []
        for source_col, frame in zip(kv_cols, extra_frames):
            conflicting = set(frame.columns) & collisions
            if conflicting:
                frame = frame.rename(
                    columns={c: f"{source_col}{sep}{c}" for c in conflicting}
                )
            renamed_frames.append(frame)
        extra_frames = renamed_frames

    df = df.drop(columns=kv_cols)
    df = pd.concat([df] + extra_frames, axis=1)
    return df

## Формирование RAW Layer

### Get raw data from all sources

In [41]:
params_store = {
    "expand": "zones,slots.zone",
    "filter": f"updated>={START_TIMESTAMP};updated<{STOP_TIMESTAMP}",
    "limit": LIMIT}
params_uom = {
    # "filter": f"updated>={START_TIMESTAMP};updated<{STOP_TIMESTAMP}",  при реальных данных сключить даты
    "limit": LIMIT}
params_product = {
    "expand": "uom,attributes.value",
    "filter": f"updated>={START_TIMESTAMP};updated<{STOP_TIMESTAMP}",
    "limit": LIMIT}
params_variant = {
    "expand": "product",
    "filter": f"updated>={START_TIMESTAMP};updated<{STOP_TIMESTAMP}",
    "limit": LIMIT}
params_agent = {
    "filter": f"updated>={START_TIMESTAMP};updated<{STOP_TIMESTAMP}",
    "limit": LIMIT}
params_in_out = {
    "expand": "positions.slot,positions.assortment,agent",
    "filter": f"updated>={START_TIMESTAMP};updated<{STOP_TIMESTAMP};applicable=true",
    "limit": LIMIT}
params_move = {
    "expand": "positions.targetSlot,positions.sourceSlot",
    "filter": f"updated>={START_TIMESTAMP};updated<{STOP_TIMESTAMP};applicable=true",
    "limit": LIMIT}


store_raw = requests.get(
    url='https://api.moysklad.ru/api/remap/1.2/entity/store', 
    headers=HEADERS, 
    params=params_store).json()

uom_raw = requests.get(
    url='https://api.moysklad.ru/api/remap/1.2/entity/uom', 
    headers=HEADERS, 
    params=params_uom).json()

product_raw = requests.get(
    url='https://api.moysklad.ru/api/remap/1.2/entity/product', 
    headers=HEADERS, 
    params=params_product).json()
variant_raw = requests.get(
    url='https://api.moysklad.ru/api/remap/1.2/entity/variant', 
    headers=HEADERS, 
    params=params_variant).json()

agent_raw = requests.get(
    url='https://api.moysklad.ru/api/remap/1.2/entity/counterparty', 
    headers=HEADERS, 
    params=params_agent).json()

demand_raw = requests.get(
    url='https://api.moysklad.ru/api/remap/1.2/entity/demand', 
    headers=HEADERS, 
    params=params_in_out).json()
supply_raw = requests.get(
    url='https://api.moysklad.ru/api/remap/1.2/entity/supply', 
    headers=HEADERS, 
    params=params_in_out).json()
loss_raw = requests.get(
    url='https://api.moysklad.ru/api/remap/1.2/entity/loss', 
    headers=HEADERS, 
    params=params_in_out).json()
enter_raw = requests.get(
    url='https://api.moysklad.ru/api/remap/1.2/entity/enter', 
    headers=HEADERS, 
    params=params_in_out).json()

move_raw = requests.get(
    url='https://api.moysklad.ru/api/remap/1.2/entity/move', 
    headers=HEADERS, 
    params=params_move).json()


### save .json file to 'temp' for Data Discovery and visual maping

In [42]:
import json

data_to_save = {
    'demand_raw.json': demand_raw,
    'variant_raw.json': variant_raw,
    'product_raw.json': product_raw,
    'store_raw.json': store_raw,
    'enter_raw.json': enter_raw,
    'move_raw.json': move_raw,
    'agent_raw.json': agent_raw,
    'uom_raw.json': uom_raw,
}

for filename, content in data_to_save.items():
    file_path = Path('temp/raw_json') / filename
    
    file_path.write_text(
        json.dumps(content, ensure_ascii=False, indent=4), 
        encoding='utf-8'
    )

## Формирование Brobze Layer

### Наборы колонок для таблиц all_docs (прообразы Bronze.operations)

In [43]:
filter_mapping_store = {
    'id':'store_id',
    'name':'name', 
    'updated':'updated', 
}

filter_mapping_zone = {
    'id':'zone_id',
    'store_id':'store_id', 
    'name':'name', 
    'updated':'updated', 
}

filter_mapping_slot = {
    'id':'slot_id',
    'store_id':'store_id', 
    'zone_id':'zone_id', 
    'name':'name', 
    'updated':'updated', 
}

filter_mapping_uom = {
    'id':'uom_id',
    'name':'name', 
    'updated':'updated', 
}

filter_mapping_product = {
    'id':'product_id',
    'name':'name', 
    'article':'article',
    'weight':'weight',
    'volume':'volume',
    'uom_id': 'uom_id',
    'Поклажедатель_id':'depositor_id',    
    'updated':'updated',
}

filter_mapping_variant = {
    'id':'variant_id',
    'product_id':'product_id', 
    'партия':'lot', 
    'дата выработки':'mfg_date',
    'barcodes':'barcodes',
    'updated':'updated', 
}

filter_mapping_agent = {
    'id':'agent_id',
    'name':'name', 
    'inn':'inn', 
    'updated':'updated', 
}

filter_mapping_in_out_move = {
    'doc_id':'doc_id',
    'doc_moment':'date', 
    'doc_name':'number',
    'agent_id':'agent_id',
    'meta_type':'type',
    'op_type':'op_type',
    'id':'prod_id',
    'quantity':'quantity', 
    'slot_id':'slot_id', 
    'doc_updated': 'updated',
}

### Store group

#### STORE TABLE

In [44]:
store_df = pd.json_normalize(
    store_raw['rows'],
    errors='ignore',
    sep='_',
)

store_df_cut = filter_and_rename_columns(store_df, filter_mapping_store)

store_df_cut

,store_id,name,updated
0,be826458-8194-11ef-0a80-08a10017d38f,Основной склад,2026-04-06 04:09:26.704


#### ZONE TABLE

In [45]:
zone_df = pd.json_normalize(
    store_raw['rows'],
    record_path=['zones', 'rows'],
    meta=['id'],
    meta_prefix='store_',
    errors='ignore',
    sep='_',
)

zone_df_cut = filter_and_rename_columns(zone_df, filter_mapping_zone)

zone_df_cut

,zone_id,store_id,name,updated
0,fe778040-d1cd-11ef-0a80-170e0035195a,be826458-8194-11ef-0a80-08a10017d38f,Стеллажи,2025-01-13 19:46:53.949
1,fe778389-d1cd-11ef-0a80-170e0035195b,be826458-8194-11ef-0a80-08a10017d38f,Приемка,2025-01-13 19:46:53.949


#### SLOT TABLE

In [46]:
slot_df = pd.json_normalize(
    store_raw['rows'],
    record_path=['slots', 'rows'],
    meta=['id'],
    meta_prefix='store_',
    errors='ignore',
    sep='_',
)

slot_df_cut = filter_and_rename_columns(slot_df, filter_mapping_slot)

slot_df_cut

,slot_id,store_id,zone_id,name,updated
0,fe77767d-d1cd-11ef-0a80-170e00351956,be826458-8194-11ef-0a80-08a10017d38f,fe778040-d1cd-11ef-0a80-170e0035195a,A1,2025-01-13 19:46:53.949
1,fe777950-d1cd-11ef-0a80-170e00351957,be826458-8194-11ef-0a80-08a10017d38f,fe778040-d1cd-11ef-0a80-170e0035195a,A2,2025-01-13 19:46:53.949
2,fe777a1b-d1cd-11ef-0a80-170e00351958,be826458-8194-11ef-0a80-08a10017d38f,fe778040-d1cd-11ef-0a80-170e0035195a,A3,2025-01-13 19:46:53.949
3,1caac6f9-3123-11f1-0a80-08660026369b,be826458-8194-11ef-0a80-08a10017d38f,fe778040-d1cd-11ef-0a80-170e0035195a,A444,2026-04-05 22:10:30.038
4,1caacb77-3123-11f1-0a80-08660026369c,be826458-8194-11ef-0a80-08a10017d38f,fe778389-d1cd-11ef-0a80-170e0035195b,A5,2026-04-05 22:10:30.038
5,fe777af6-d1cd-11ef-0a80-170e00351959,be826458-8194-11ef-0a80-08a10017d38f,fe778389-d1cd-11ef-0a80-170e0035195b,B,2025-01-13 19:46:53.949


### UOM TABLE

In [47]:
uom_df = pd.json_normalize(
    uom_raw['rows'],
    errors='ignore',
    sep='_',
)

uom_df_cut = filter_and_rename_columns(uom_df, filter_mapping_uom)

uom_df_cut

,uom_id,name,updated
0,061721df-9197-49a5-b637-7f5b4d3be969,дюйм,2012-11-02 12:07:44.658
1,0dd4fe8b-e59e-486e-bde5-b52fe0e25415,мес,2012-11-02 12:07:44.845
2,151af5a2-3df9-4aca-851c-814c8b3a65e6,ц,2012-11-02 12:07:44.813
3,1700dfba-e9e7-4c98-9857-8d984ab48b2b,ч,2012-11-02 12:07:44.827
4,19f1edc0-fc42-4001-94cb-c9ec9c62ec10,шт,2012-11-02 12:07:44.874
5,1b791118-1829-4c30-82fb-e0750d18bd71,мм,2012-11-02 12:07:44.590
6,1e1facff-ad04-48c3-8d07-54ff9cd65135,г; лет,2012-11-02 12:07:44.858
7,2177a357-1bff-45df-b6ae-7f5109b8ca51,миля,2012-11-02 12:07:44.672
8,24d6874a-37a9-4b74-a0b5-3cd7da742441,гг,2012-11-02 12:07:44.771
9,250d5161-f815-4524-85b6-fd6afebc1ad6,л; дм3,2012-11-02 12:07:44.732


### Goods group

#### PRODUCT TABLE

In [48]:
product_df = json_normalize_extended(
    product_raw['rows'],
    errors='ignore',
    sep='_',
    kv_name_key="name",
    kv_value_key="value",
)

product_df_cut = filter_and_rename_columns(product_df, filter_mapping_product)

product_df_cut

,product_id,name,article,weight,volume,uom_id,updated
0,b0b826ee-d1f4-11ef-0a80-09d1000028b1,Килька балтийская копченая в томатном соусе 175 г.,A1111111111111,0.0,990.0,19f1edc0-fc42-4001-94cb-c9ec9c62ec10,2026-04-06 22:00:11.495
1,bfff99bc-3121-11f1-0a80-0782002603bb,Сартина тихоокеанская в банке,B3333333333,6.0,800.0,19f1edc0-fc42-4001-94cb-c9ec9c62ec10,2026-04-06 22:00:25.799


#### VARIANTS TABLE

In [49]:
variant_df = json_normalize_extended(
    variant_raw['rows'],
    errors='ignore',
    sep='_',
    kv_name_key="name",
    kv_value_key="value",
)

variant_df_cut = filter_and_rename_columns(variant_df, filter_mapping_variant)

variant_df_cut

,variant_id,product_id,lot,mfg_date,barcodes,updated
0,4aae53c6-3122-11f1-0a80-174e00261815,b0b826ee-d1f4-11ef-0a80-09d1000028b1,44.88,12.06.25,[{'ean13': '2000000000169'}],2026-04-05 22:04:37.677
1,4ab1ac38-3122-11f1-0a80-174e0026181b,b0b826ee-d1f4-11ef-0a80-09d1000028b1,44.88,29.09.26,[{'ean13': '2000000000176'}],2026-04-05 22:04:37.700
2,4ab3f31e-3122-11f1-0a80-174e00261821,b0b826ee-d1f4-11ef-0a80-09d1000028b1,55.98,12.06.25,[{'ean13': '2000000000183'}],2026-04-05 22:04:37.714
3,4ab63158-3122-11f1-0a80-174e00261827,b0b826ee-d1f4-11ef-0a80-09d1000028b1,55.98,29.09.26,[{'ean13': '2000000000190'}],2026-04-05 22:04:37.729
4,faa2676d-3121-11f1-0a80-1a9d0025bfc1,bfff99bc-3121-11f1-0a80-0782002603bb,34.25,12.02.25,[{'ean13': '2000000000107'}],2026-04-05 22:02:23.381
5,faa6e34e-3121-11f1-0a80-1a9d0025bfc7,bfff99bc-3121-11f1-0a80-0782002603bb,34.25,14.05.26,[{'ean13': '2000000000114'}],2026-04-05 22:02:23.412
6,faaa75e6-3121-11f1-0a80-1a9d0025bfcd,bfff99bc-3121-11f1-0a80-0782002603bb,45.78,12.02.25,[{'ean13': '2000000000121'}],2026-04-05 22:02:23.435
7,faadfb10-3121-11f1-0a80-1a9d0025bfd3,bfff99bc-3121-11f1-0a80-0782002603bb,45.78,14.05.26,[{'ean13': '2000000000138'}],2026-04-05 22:02:23.458
8,fab14ce8-3121-11f1-0a80-1a9d0025bfd9,bfff99bc-3121-11f1-0a80-0782002603bb,56.98,12.02.25,[{'ean13': '2000000000145'}],2026-04-05 22:02:23.480
9,fab3bb25-3121-11f1-0a80-1a9d0025bfdf,bfff99bc-3121-11f1-0a80-0782002603bb,56.98,14.05.26,[{'ean13': '2000000000152'}],2026-04-05 22:02:23.496


### AGENT TABLE

In [50]:
agent_df = pd.json_normalize(
    agent_raw['rows'],
    errors='ignore',
    sep='_',
)

agent_df_cut = filter_and_rename_columns(agent_df, filter_mapping_agent)

agent_df_cut

,agent_id,name,inn,updated
0,64b4e652-3171-11f1-0a80-18c5002780ed,Котрагент Тестович,8966578996,2026-04-06 07:32:45.490


### IN_OUT TABLE

 ---
 
 #### DEMAND (Отгрузка)
 Товар уходит со склада к контрагенту. Ячейка освобождается.

 meta_type позиции = 'demandposition'

 ---

In [58]:
demand_df = json_normalize_extended(
    demand_raw['rows'],
    record_path=['positions', 'rows'],
    meta=['id', 'name', 'moment', 'updated', 'agent'],
    meta_prefix='doc_',
    errors='ignore',
    sep='_',
    kv_name_key="name",
    kv_value_key="value",
)
demand_df['agent_id']=demand_df['doc_agent'].str.get('id') # !!! если записей на соответствующую дату нет, то код тут упадет с ошибкой
demand_df['op_type'] = 'out'
demand_df_cut = filter_and_rename_columns(demand_df, filter_mapping_in_out_move)

demand_df_cut


,doc_id,date,number,agent_id,type,op_type,prod_id,quantity,slot_id,updated
0,8becde9a-3145-11f1-0a80-01b100251788,2026-04-06 02:16:00.000,00001,be82bd0a-8194-11ef-0a80-08a10017d392,demandposition,out,8bece5b5-3145-11f1-0a80-01b100251789,50.0,fe77767d-d1cd-11ef-0a80-170e00351956,2026-04-06 02:20:08.825
1,8becde9a-3145-11f1-0a80-01b100251788,2026-04-06 02:16:00.000,00001,be82bd0a-8194-11ef-0a80-08a10017d392,demandposition,out,8bece7dd-3145-11f1-0a80-01b10025178a,10.0,fe777950-d1cd-11ef-0a80-170e00351957,2026-04-06 02:20:08.825
2,b4472725-3145-11f1-0a80-195d0025aafb,2026-04-06 02:17:00.000,00002,be82bd0a-8194-11ef-0a80-08a10017d392,demandposition,out,260edfe0-3146-11f1-0a80-195d0025ab60,50.0,fe777950-d1cd-11ef-0a80-170e00351957,2026-04-06 02:21:18.170
3,b4472725-3145-11f1-0a80-195d0025aafb,2026-04-06 02:17:00.000,00002,be82bd0a-8194-11ef-0a80-08a10017d392,demandposition,out,260ee53f-3146-11f1-0a80-195d0025ab61,300.0,1caac6f9-3123-11f1-0a80-08660026369b,2026-04-06 02:21:18.170
4,bf67dd0b-3146-11f1-0a80-174e002694d2,2026-04-06 02:25:00.000,00003,be829072-8194-11ef-0a80-08a10017d390,demandposition,out,bf67eb1c-3146-11f1-0a80-174e002694d3,6.0,NaN,2026-04-06 02:25:35.374


 ---

 #### SUPPLY (Приёмка)
 Товар приходит на склад от поставщика. Ячейка занимается.

 meta_type позиции = 'supplyposition'

 ---

In [52]:
supply_df = json_normalize_extended(
    supply_raw['rows'],
    record_path=['positions', 'rows'],
    meta=['id', 'name', 'moment', 'updated', 'agent'],
    meta_prefix='doc_',
    errors='ignore',
    sep='_',
    kv_name_key="name",
    kv_value_key="value",
)
supply_df['agent_id']=supply_df['doc_agent'].str.get('id')
supply_df['op_type'] = 'in'
supply_df_cut = filter_and_rename_columns(supply_df, filter_mapping_in_out_move)

supply_df_cut


,doc_id,date,number,agent_id,type,op_type,prod_id,quantity,slot_id,updated
0,41b38baa-3145-11f1-0a80-1c750025cadc,2026-04-06 02:13:00.000,00001,be82bd0a-8194-11ef-0a80-08a10017d392,supplyposition,in,41b39336-3145-11f1-0a80-1c750025cadd,50.0,fe77767d-d1cd-11ef-0a80-170e00351956,2026-04-06 02:14:54.953
1,41b38baa-3145-11f1-0a80-1c750025cadc,2026-04-06 02:13:00.000,00001,be82bd0a-8194-11ef-0a80-08a10017d392,supplyposition,in,41b3967e-3145-11f1-0a80-1c750025cade,70.0,fe777950-d1cd-11ef-0a80-170e00351957,2026-04-06 02:14:54.953
2,41b38baa-3145-11f1-0a80-1c750025cadc,2026-04-06 02:13:00.000,00001,be82bd0a-8194-11ef-0a80-08a10017d392,supplyposition,in,41b397b1-3145-11f1-0a80-1c750025cadf,30.0,fe777a1b-d1cd-11ef-0a80-170e00351958,2026-04-06 02:14:54.953
3,68f698ae-3145-11f1-0a80-18c50026787b,2026-04-06 02:15:00.000,00002,be82bd0a-8194-11ef-0a80-08a10017d392,supplyposition,in,68f6a28a-3145-11f1-0a80-18c50026787c,900.0,1caac6f9-3123-11f1-0a80-08660026369b,2026-04-06 02:16:00.844


 ---

 #### LOSS (Списание)

 Товар выбывает со склада без отгрузки поклажедателю (брак, порча, недостача). Ячейка освобождается.

 meta_type позиции = 'lossposition'

 ---

In [53]:
loss_df = json_normalize_extended(
    loss_raw['rows'],
    record_path=['positions', 'rows'],
    meta=['id', 'name', 'moment', 'updated', 'agent'],
    meta_prefix='doc_',
    errors='ignore',
    sep='_',
    kv_name_key="name",
    kv_value_key="value",
)
loss_df['agent_id']=loss_df['doc_agent'].str.get('id')
loss_df['op_type'] = 'in'
loss_df_cut = filter_and_rename_columns(loss_df, filter_mapping_in_out_move)

loss_df_cut


,doc_id,date,number,agent_id,type,op_type,prod_id,quantity,updated
0,9e503078-3146-11f1-0a80-1c750025cc91,2026-04-06 02:24:00.000,00001,NaN,lossposition,in,9e5035ce-3146-11f1-0a80-1c750025cc92,35.0,2026-04-06 02:24:39.857


 ---

 #### ENTER (Оприходование)

 Товар появляется на складе без входящей накладной (излишки, начальные остатки). Ячейка занимается.

 meta_type позиции = 'enterposition'

 ---

In [54]:
enter_df = json_normalize_extended(
    enter_raw['rows'],
    record_path=['positions', 'rows'],
    meta=['id', 'name', 'moment', 'updated', 'agent'],
    meta_prefix='doc_',
    errors='ignore',
    sep='_',
    kv_name_key="name",
    kv_value_key="value",
)
enter_df['agent_id']=enter_df['doc_agent'].str.get('id')
enter_df['op_type'] = 'in'
enter_df_cut = filter_and_rename_columns(enter_df, filter_mapping_in_out_move)

enter_df_cut


,doc_id,date,number,agent_id,type,op_type,prod_id,quantity,slot_id,updated
0,848d50d6-3146-11f1-0a80-078200269927,2026-04-06 02:23:00.000,00001,NaN,enterposition,in,848d578c-3146-11f1-0a80-078200269928,15.0,fe777950-d1cd-11ef-0a80-170e00351957,2026-04-06 02:23:56.630


 ---

 #### MOVE (Перемещение)

 Товар переезжает из одной ячейки в другую внутри склада (или между складами).

 Одна операция перемещения порождает ДВЕ строки:

- moveout — товар покидает sourceSlot (ячейка освобождается)

  `slot_id = sourceSlot_id`

- movein  — товар занимает targetSlot (ячейка занимается)

  `slot_id = targetSlot_id`

 ---

In [55]:
move_df_base = json_normalize_extended(
    move_raw['rows'],
    record_path=['positions', 'rows'],
    meta=['id', 'name', 'moment', 'updated'],
    meta_prefix='doc_',
    errors='ignore',
    sep='_',
    kv_name_key="name",
    kv_value_key="value",
)


# --- строки MOVEOUT: ячейка-источник ---
moveout_df = move_df_base.copy()
moveout_df['slot_id'] = moveout_df.get('sourceSlot_id')  # откуда ушёл
moveout_df['op_type'] = 'out'

# --- строки MOVEIN: ячейка-назначение ---
movein_df = move_df_base.copy()
movein_df['slot_id'] = movein_df.get('targetSlot_id')    # куда пришёл
movein_df['op_type'] = 'in'

# Объединяем и обрезаем оба направления; итоговый df содержит по 2 строки на каждую позицию
move_df = pd.concat([
    moveout_df,
    movein_df,
], ignore_index=True)
move_df_cut = filter_and_rename_columns(move_df, filter_mapping_in_out_move)

move_df_cut

,doc_id,date,number,type,op_type,prod_id,quantity,slot_id,updated
0,4a1ff0d7-3146-11f1-0a80-01b100251857,2026-04-06 02:21:00.000,00001,moveposition,out,4a1ffa72-3146-11f1-0a80-01b100251858,550.0,1caac6f9-3123-11f1-0a80-08660026369b,2026-04-06 02:22:18.651
1,4a1ff0d7-3146-11f1-0a80-01b100251857,2026-04-06 02:21:00.000,00001,moveposition,in,4a1ffa72-3146-11f1-0a80-01b100251858,550.0,1caacb77-3123-11f1-0a80-08660026369c,2026-04-06 02:22:18.651


 ---

 #### UNIFIED: in_out_docs

 Объединяем все 5 in_out документов в единый датафрейм.

 Прообраз таблицы в BRONZE.operations.

 ---

In [56]:
# При union сохраняется структура, заданная в каждой отдельной таблице согласно параметру filter_mapping_in_out_move. 
# Сортировка по документу-времени — удобно для визуального контроля и будущего Bronze.
in_out_df = (
    pd.concat([
        demand_df_cut,
        supply_df_cut,
        enter_df_cut,
        loss_df_cut,
        move_df_cut,
    ], ignore_index=True)
    .sort_values(['date', 'number'])
    .reset_index(drop=True)
    .query("slot_id.notna()")
)

in_out_df

,doc_id,date,number,agent_id,type,op_type,prod_id,quantity,slot_id,updated
0,41b38baa-3145-11f1-0a80-1c750025cadc,2026-04-06 02:13:00.000,00001,be82bd0a-8194-11ef-0a80-08a10017d392,supplyposition,in,41b39336-3145-11f1-0a80-1c750025cadd,50.0,fe77767d-d1cd-11ef-0a80-170e00351956,2026-04-06 02:14:54.953
1,41b38baa-3145-11f1-0a80-1c750025cadc,2026-04-06 02:13:00.000,00001,be82bd0a-8194-11ef-0a80-08a10017d392,supplyposition,in,41b3967e-3145-11f1-0a80-1c750025cade,70.0,fe777950-d1cd-11ef-0a80-170e00351957,2026-04-06 02:14:54.953
2,41b38baa-3145-11f1-0a80-1c750025cadc,2026-04-06 02:13:00.000,00001,be82bd0a-8194-11ef-0a80-08a10017d392,supplyposition,in,41b397b1-3145-11f1-0a80-1c750025cadf,30.0,fe777a1b-d1cd-11ef-0a80-170e00351958,2026-04-06 02:14:54.953
3,68f698ae-3145-11f1-0a80-18c50026787b,2026-04-06 02:15:00.000,00002,be82bd0a-8194-11ef-0a80-08a10017d392,supplyposition,in,68f6a28a-3145-11f1-0a80-18c50026787c,900.0,1caac6f9-3123-11f1-0a80-08660026369b,2026-04-06 02:16:00.844
4,8becde9a-3145-11f1-0a80-01b100251788,2026-04-06 02:16:00.000,00001,be82bd0a-8194-11ef-0a80-08a10017d392,demandposition,out,8bece5b5-3145-11f1-0a80-01b100251789,50.0,fe77767d-d1cd-11ef-0a80-170e00351956,2026-04-06 02:20:08.825
5,8becde9a-3145-11f1-0a80-01b100251788,2026-04-06 02:16:00.000,00001,be82bd0a-8194-11ef-0a80-08a10017d392,demandposition,out,8bece7dd-3145-11f1-0a80-01b10025178a,10.0,fe777950-d1cd-11ef-0a80-170e00351957,2026-04-06 02:20:08.825
6,b4472725-3145-11f1-0a80-195d0025aafb,2026-04-06 02:17:00.000,00002,be82bd0a-8194-11ef-0a80-08a10017d392,demandposition,out,260edfe0-3146-11f1-0a80-195d0025ab60,50.0,fe777950-d1cd-11ef-0a80-170e00351957,2026-04-06 02:21:18.170
7,b4472725-3145-11f1-0a80-195d0025aafb,2026-04-06 02:17:00.000,00002,be82bd0a-8194-11ef-0a80-08a10017d392,demandposition,out,260ee53f-3146-11f1-0a80-195d0025ab61,300.0,1caac6f9-3123-11f1-0a80-08660026369b,2026-04-06 02:21:18.170
8,4a1ff0d7-3146-11f1-0a80-01b100251857,2026-04-06 02:21:00.000,00001,NaN,moveposition,out,4a1ffa72-3146-11f1-0a80-01b100251858,550.0,1caac6f9-3123-11f1-0a80-08660026369b,2026-04-06 02:22:18.651
9,4a1ff0d7-3146-11f1-0a80-01b100251857,2026-04-06 02:21:00.000,00001,NaN,moveposition,in,4a1ffa72-3146-11f1-0a80-01b100251858,550.0,1caacb77-3123-11f1-0a80-08660026369c,2026-04-06 02:22:18.651


### save .csv file to 'temp' for Data Discovery and visual maping

In [57]:
demand_df.to_csv(Path('temp/df_csv/demand.csv'))
supply_df.to_csv(Path('temp/df_csv/supply.csv'))
loss_df.to_csv(Path('temp/df_csv/loss.csv'))
enter_df.to_csv(Path('temp/df_csv/enter.csv'))
move_df_base.to_csv(Path('temp/df_csv/move.csv'))
in_out_df.to_csv(Path('temp/df_csv/in_out.csv'))
store_df.to_csv(Path('temp/df_csv/store.csv'))
slot_df.to_csv(Path('temp/df_csv/slot.csv'))
zone_df.to_csv(Path('temp/df_csv/zone.csv'))
product_df.to_csv(Path('temp/df_csv/product.csv'))
variant_df.to_csv(Path('temp/df_csv/variant.csv'))
agent_df.to_csv(Path('temp/df_csv/agent.csv'))